# Benchmark Diagnostics

Investigate benchmark learning curves, checkpoint drift, pair-ranking behavior, confident errors, and connected-component clustering failures.

Pair-level analysis defaults to `nc_voters_changed` so large benchmarks do not create unnecessarily large diagnostic tables.

In [ ]:
import hashlib
import json
import os
import resource
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import schedulefree
import torch
from IPython.display import display
from sklearn.metrics import (
    adjusted_rand_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

from identitypfn.data_loader import SyntheticWorldDataLoader
from identitypfn.model import NanoERPFNLinker, NanoERPFNModel
from identitypfn.dgp.people.simple import ObservationConfig, generate_world
from identitypfn.tokenizer import Tokenizer
from identitypfn.train import load_benchmark_frame, load_benchmarks
from identitypfn.utils import get_default_device, set_randomness_seed

RANDOM_SEED = 1337
set_randomness_seed(RANDOM_SEED)
device = get_default_device()
print(f"device: {device}")

## Configuration

In [ ]:
BENCHMARK_NAMES = [
    # "nc_voters_shared",
    "nc_voters_changed",
    # "fodors_zagats",
    # "dblp_acm",
    # "dblp_google_scholar",
    "amazon_google_price_numeric",
    # "amazon_google_price_both",
    # "walmart_amazon",
    # "bpid_matching_balanced",
    # "ice_id_people_200",
]
CROSS_SOURCE_BENCHMARKS = {
    # "fodors_zagats",
    # "dblp_acm",
    # "dblp_google_scholar",
    "amazon_google_price_numeric",
    # "amazon_google_price_both",
    # "walmart_amazon",
}
PAIR_ANALYSIS_BENCHMARK = "nc_voters_changed"

FROZEN_DGP_CONFIGS = {
    "person_dgp": {
        "observation_config": ObservationConfig(
            missing_rate=0.1, nickname_rate=0.05, corruption_rate=0.1
        ),
        "seed": 502,
    },
}


In [ ]:
EMBEDDING_SIZE = 96
NUM_ATTENTION_HEADS = 4
MLP_HIDDEN_SIZE = 192
NUM_LAYERS = 3
MAX_CATEGORIES = 512

# RECORD_REPRESENTATION options: "mean_pool", "entity_target_column"
RECORD_REPRESENTATION = "entity_target_column"
# ADJACENCY_DECODER options: "pair_mlp", "slot_coassignment", "slot_pair_mlp"
ADJACENCY_DECODER = "pair_mlp"
ENTITY_SLOT_COUNT = 300
NUM_SLOT_ATTENTION_LAYERS = 1

NUM_STEPS = 3000
EVAL_EVERY = 250
BATCH_SIZE = 16
LEARNING_RATE = 4e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0

# POS_WEIGHT = 20.0
POS_WEIGHT = None

# TEXT_BACKEND = "fasttext"
TEXT_BACKEND = "sentence_transformer"
IDENTIFIER_BACKEND = "fasttext"
NORMALIZE_IDENTIFIERS = True
DEFAULT_PHONE_REGION = "US"
FROZEN_DGP_N_RECORDS = 250
FROZEN_DGP_P_MATCH = 0.05
FROZEN_DGP_N_FIELDS = 6
FROZEN_DGP_SCHEMA_TEMPERATURE = 1.0
MANIFEST_PATH = Path("benchmark_data/manifest.json")
RESULTS_DIRECTORY = Path("results/benchmark_runs")
CHECKPOINT_DIRECTORY = Path("results/model_checkpoints")
SAVE_MODELS = True
SAVE_BEST_MODEL = True
SAVE_LAST_MODEL = True
PAIR_ANALYSIS_CHECKPOINT_PATH = None
RESUME_CHECKPOINT_PATH = None
# RESUME_CHECKPOINT_PATH = Path(
#     "results/model_checkpoints/20260724_052550_seed1337_bf6bef2c_latest.pt"
# )

TRAIN_GENERATOR = "wag"
WAG_ALLOW_OLLAMA = False
WAG_OLLAMA_MODEL = "llama3.2:1b"
HARD_NEGATIVE_RATE = 0.0
HARD_NEGATIVE_CONTRACT_WEIGHTS = None

RUN_CONFIG = {
    "random_seed": RANDOM_SEED,
    "num_steps": NUM_STEPS,
    "eval_every": EVAL_EVERY,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "AdamWScheduleFree",
    "weight_decay": WEIGHT_DECAY,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "pos_weight": POS_WEIGHT,
    "embedding_size": EMBEDDING_SIZE,
    "num_attention_heads": NUM_ATTENTION_HEADS,
    "mlp_hidden_size": MLP_HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "max_categories": MAX_CATEGORIES,
    "record_representation": RECORD_REPRESENTATION,
    "adjacency_decoder": ADJACENCY_DECODER,
    "entity_slot_count": ENTITY_SLOT_COUNT,
    "num_slot_attention_layers": NUM_SLOT_ATTENTION_LAYERS,
    "text_backend": TEXT_BACKEND,
    "identifier_backend": IDENTIFIER_BACKEND,
    "normalize_identifiers": NORMALIZE_IDENTIFIERS,
    "default_phone_region": DEFAULT_PHONE_REGION,
    "train_generator": TRAIN_GENERATOR,
    "wag_allow_ollama": WAG_ALLOW_OLLAMA,
    "wag_ollama_model": WAG_OLLAMA_MODEL,
    "hard_negative_rate": HARD_NEGATIVE_RATE,
    "hard_negative_contract_weights": HARD_NEGATIVE_CONTRACT_WEIGHTS,
    "benchmark_names": BENCHMARK_NAMES,
    "frozen_dgp_n_records": FROZEN_DGP_N_RECORDS,
    "frozen_dgp_p_match": FROZEN_DGP_P_MATCH,
    "frozen_dgp_n_fields": FROZEN_DGP_N_FIELDS,
    "frozen_dgp_schema_temperature": FROZEN_DGP_SCHEMA_TEMPERATURE,
    "save_models": SAVE_MODELS,
    "save_best_model": SAVE_BEST_MODEL,
    "save_last_model": SAVE_LAST_MODEL,
    "resume_checkpoint_path": str(RESUME_CHECKPOINT_PATH)
    if RESUME_CHECKPOINT_PATH is not None
    else None,
    "frozen_dgp_configs": {
        name: {
            "missing_rate": config["observation_config"].missing_rate,
            "nickname_rate": config["observation_config"].nickname_rate,
            "corruption_rate": config["observation_config"].corruption_rate,
            "seed": config["seed"],
        }
        for name, config in FROZEN_DGP_CONFIGS.items()
    },
}
RUN_STARTED_AT = datetime.now().astimezone()
RUN_CONFIG_JSON = json.dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":"))
RUN_CONFIG_HASH = hashlib.sha256(RUN_CONFIG_JSON.encode()).hexdigest()[:8]
RUN_ID = f"{RUN_STARTED_AT:%Y%m%d_%H%M%S}_seed{RANDOM_SEED}_{RUN_CONFIG_HASH}"
print(f"run_id: {RUN_ID}")

tokenizer = Tokenizer(
    text_backend=TEXT_BACKEND,
    identifier_backend=IDENTIFIER_BACKEND,
    normalize_identifiers=NORMALIZE_IDENTIFIERS,
    default_phone_region=DEFAULT_PHONE_REGION,
)
benchmarks = load_benchmarks(MANIFEST_PATH, names=BENCHMARK_NAMES)

with MANIFEST_PATH.open() as file:
    manifest_config = json.load(file)

extra_benchmarks = []
for benchmark in benchmarks:
    config = manifest_config[benchmark["name"]]
    metadata_columns = [*config["field_types"].keys()]
    ignored_columns = config.get("ignored_columns", [])
    if "source_table" in ignored_columns:
        metadata_columns.insert(0, "source_table")
    if benchmark["name"] == "bpid_matching_balanced":
        metadata_columns = [
            "pair_id",
            "source_table",
            "match_label",
            *config["field_types"].keys(),
        ]

    metadata_frame = load_benchmark_frame(MANIFEST_PATH, config, metadata_columns)
    if (
        benchmark["name"] in CROSS_SOURCE_BENCHMARKS
        and "source_table" in metadata_frame
    ):
        benchmark["source_table_values"] = metadata_frame["source_table"].to_numpy()
    else:
        benchmark["source_table_values"] = None

    if benchmark["name"] == "bpid_matching_balanced":
        paired_indices = []
        paired_targets = []
        for _, group in metadata_frame.groupby("pair_id", sort=False):
            if (
                set(group["source_table"]) != {"profile1", "profile2"}
                or len(group) != 2
            ):
                continue
            left_index = int(group.index[group["source_table"] == "profile1"][0])
            right_index = int(group.index[group["source_table"] == "profile2"][0])
            paired_indices.append(
                (min(left_index, right_index), max(left_index, right_index))
            )
            paired_targets.append(str(group["match_label"].iloc[0]).lower() == "true")

        paired_benchmark = benchmark.copy()
        paired_benchmark["name"] = "bpid_matching_paired"
        paired_benchmark["description"] = (
            "BPID labeled person-profile matching pairs evaluated on the original "
            "profile1/profile2 candidate pairs."
        )
        paired_benchmark["pair_scope"] = "paired_rows"
        paired_benchmark["paired_row_indices"] = np.asarray(paired_indices, dtype=int)
        paired_benchmark["paired_row_targets"] = np.asarray(paired_targets, dtype=bool)
        paired_benchmark["source_table_values"] = None
        extra_benchmarks.append(paired_benchmark)

benchmarks.extend(extra_benchmarks)


def pair_targets(benchmark):
    entity_ids = benchmark["entity_ids"]
    true_adjacency = entity_ids[:, None] == entity_ids[None, :]
    pair_mask = np.triu(np.ones_like(true_adjacency, dtype=bool), k=1)
    pair_scope = "all_pairs"
    excluded_same_source_pairs = 0

    if benchmark.get("pair_scope") == "paired_rows":
        pair_mask = np.zeros_like(true_adjacency, dtype=bool)
        pair_indices = benchmark["paired_row_indices"]
        pair_mask[pair_indices[:, 0], pair_indices[:, 1]] = True
        return (
            benchmark["paired_row_targets"].astype(bool),
            pair_mask,
            "paired_rows",
            0,
        )

    if benchmark["name"] in CROSS_SOURCE_BENCHMARKS:
        source_values = benchmark.get("source_table_values")
        if source_values is not None:
            cross_source_mask = source_values[:, None] != source_values[None, :]
            excluded_same_source_pairs = int((pair_mask & ~cross_source_mask).sum())
            pair_mask &= cross_source_mask
            pair_scope = "cross_source"

    return true_adjacency[pair_mask], pair_mask, pair_scope, excluded_same_source_pairs


def world_to_benchmark(name, world):
    return {
        "name": name,
        "description": "Frozen synthetic person-DGP sniff-test world.",
        "kind": "single_table",
        "entity_sample": None,
        "records": world.records,
        "field_types": world.field_types,
        "entity_ids": world.entity_ids,
    }


for name, config in FROZEN_DGP_CONFIGS.items():
    world = generate_world(
        n_records=FROZEN_DGP_N_RECORDS,
        p_match=FROZEN_DGP_P_MATCH,
        n_fields=FROZEN_DGP_N_FIELDS,
        schema_temperature=FROZEN_DGP_SCHEMA_TEMPERATURE,
        **config,
    )
    benchmarks.append(world_to_benchmark(name, world))

benchmark_by_name = {benchmark["name"]: benchmark for benchmark in benchmarks}


def make_model():
    return NanoERPFNModel(
        embedding_size=EMBEDDING_SIZE,
        text_embedding_size=tokenizer.text_embedding_dim,
        identifier_embedding_size=tokenizer.identifier_embedding_dim,
        num_attention_heads=NUM_ATTENTION_HEADS,
        mlp_hidden_size=MLP_HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        max_categories=MAX_CATEGORIES,
        record_representation=RECORD_REPRESENTATION,
        adjacency_decoder=ADJACENCY_DECODER,
        entity_slot_count=ENTITY_SLOT_COUNT,
        num_slot_attention_layers=NUM_SLOT_ATTENTION_LAYERS,
    ).to(device)


def benchmark_overview(benchmark):
    records = benchmark["records"]
    entity_ids = benchmark["entity_ids"]
    targets, pair_mask, pair_scope, excluded_same_source_pairs = pair_targets(benchmark)
    candidate_left, candidate_right = np.where(pair_mask)
    positive_indices = np.flatnonzero(targets.astype(bool))
    if len(positive_indices):
        left = candidate_left[positive_indices]
        right = candidate_right[positive_indices]
        equal_cells = records.iloc[left].reset_index(drop=True).eq(
            records.iloc[right].reset_index(drop=True)
        ) | (
            records.iloc[left].reset_index(drop=True).isna()
            & records.iloc[right].reset_index(drop=True).isna()
        )
        exact_positive_pair_rate = float(equal_cells.all(axis=1).mean())
    else:
        exact_positive_pair_rate = np.nan
    return {
        "benchmark": benchmark["name"],
        "records": len(records),
        "entities": len(np.unique(entity_ids)),
        "fields": records.shape[1],
        "pair_scope": pair_scope,
        "candidate_pairs": int(targets.size),
        "excluded_same_source_pairs": excluded_same_source_pairs,
        "prevalence": float(targets.mean()),
        "missing_cell_rate": float(records.isna().to_numpy().mean()),
        "exact_positive_pair_rate": exact_positive_pair_rate,
        "kind": benchmark["kind"],
        "field_types": ", ".join(benchmark["field_types"]),
        "description": benchmark["description"],
    }


display(
    pd.DataFrame([benchmark_overview(benchmark) for benchmark in benchmarks])
    .set_index("benchmark")
    .sort_index()
)

## Checkpointed Training

Evaluate every benchmark separately at each checkpoint and retain model states for later pair-level comparison.

In [ ]:
def move_to_device(values):
    return {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in values.items()
    }


def current_rss_mb():
    usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if os.uname().sysname == "Darwin":
        return usage / (1024 * 1024)
    return usage / 1024


def current_device_memory_mb():
    torch_device = torch.device(device)
    if torch_device.type == "cuda" and torch.cuda.is_available():
        return torch.cuda.max_memory_allocated(torch_device) / (1024 * 1024)
    if torch_device.type == "mps" and hasattr(torch.mps, "current_allocated_memory"):
        return torch.mps.current_allocated_memory() / (1024 * 1024)
    return None


def batch_shape_summary(full_data):
    record_counts = [len(records) for records in full_data["records"]]
    field_counts = [records.shape[1] for records in full_data["records"]]
    return {
        "batch_records_min": min(record_counts),
        "batch_records_max": max(record_counts),
        "batch_records_mean": float(np.mean(record_counts)),
        "batch_fields_min": min(field_counts),
        "batch_fields_max": max(field_counts),
        "batch_fields_mean": float(np.mean(field_counts)),
    }


def clone_state_dict(model):
    return {
        key: value.detach().cpu().clone() for key, value in model.state_dict().items()
    }


def evaluate_paper_aligned_benchmark(linker, benchmark):
    entity_ids = benchmark["entity_ids"]
    predicted_labels = linker.fit_predict(
        benchmark["records"], benchmark["field_types"]
    )
    targets, pair_mask, pair_scope, excluded_same_source_pairs = pair_targets(benchmark)
    probabilities = linker.adjacency_proba_[pair_mask]
    predictions = probabilities >= linker.threshold

    row = {
        "name": benchmark["name"],
        "n_records": len(entity_ids),
        "n_entities": len(np.unique(entity_ids)),
        "n_fields": benchmark["records"].shape[1],
        "pair_scope": pair_scope,
        "candidate_pairs": int(targets.size),
        "excluded_same_source_pairs": excluded_same_source_pairs,
        "prevalence": float(targets.mean()),
    }
    if np.unique(targets).size < 2:
        row.update(
            {
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "pair_precision": np.nan,
                "pair_recall": np.nan,
                "pair_f1": np.nan,
                "best_pair_f1": np.nan,
                "best_pair_threshold": np.nan,
                "adjusted_rand": np.nan,
            }
        )
        return row

    threshold_precision, threshold_recall, thresholds = precision_recall_curve(
        targets, probabilities
    )
    threshold_f1 = (
        2
        * threshold_precision[:-1]
        * threshold_recall[:-1]
        / np.maximum(threshold_precision[:-1] + threshold_recall[:-1], 1e-12)
    )
    best_threshold_index = int(np.argmax(threshold_f1))

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="The number of unique classes is greater than 50%.*",
            category=UserWarning,
        )
        adjusted_rand = (
            np.nan
            if pair_scope == "paired_rows"
            else adjusted_rand_score(entity_ids, predicted_labels)
        )

    row.update(
        {
            "roc_auc": roc_auc_score(targets, probabilities),
            "pr_auc": average_precision_score(targets, probabilities),
            "pair_precision": precision_score(targets, predictions, zero_division=0),
            "pair_recall": recall_score(targets, predictions, zero_division=0),
            "pair_f1": f1_score(targets, predictions, zero_division=0),
            "best_pair_f1": float(threshold_f1[best_threshold_index]),
            "best_pair_threshold": float(thresholds[best_threshold_index]),
            "adjusted_rand": adjusted_rand,
        }
    )
    return row


def checkpoint_frame(model, step):
    linker = NanoERPFNLinker(model, tokenizer, device=device)
    frame = pd.DataFrame(
        [
            evaluate_paper_aligned_benchmark(linker, benchmark)
            for benchmark in benchmarks
        ]
    )
    frame.insert(0, "step", step)
    return frame


def with_run_metadata(frame, run_status, last_completed_step):
    frame = frame.copy()
    frame.insert(0, "run_id", RUN_ID)
    frame.insert(1, "run_started_at", RUN_STARTED_AT.isoformat())
    frame.insert(2, "config_hash", RUN_CONFIG_HASH)
    frame.insert(3, "run_status", run_status)
    frame.insert(4, "last_completed_step", last_completed_step)
    for key, value in reversed(RUN_CONFIG.items()):
        serialized_value = (
            json.dumps(value, sort_keys=True, separators=(",", ":"))
            if isinstance(value, (dict, list))
            else value
        )
        frame.insert(5, key, serialized_value)
    return frame


def build_run_results(run_status, last_completed_step):
    history_frame = pd.concat(checkpoint_frames, ignore_index=True)
    loss_frame = pd.DataFrame(training_losses)
    if loss_frame.empty:
        loss_frame = pd.DataFrame({"step": []})
    profile_frame = pd.DataFrame(profile_history)
    if profile_frame.empty:
        profile_frame = pd.DataFrame({"step": []})
    results = history_frame.merge(loss_frame, on="step", how="left")
    results = results.merge(profile_frame, on="step", how="left")
    return with_run_metadata(results, run_status, last_completed_step)


def build_profile_results(run_status, last_completed_step):
    profile_frame = pd.DataFrame(profile_history)
    if profile_frame.empty:
        profile_frame = pd.DataFrame({"step": []})
    loss_frame = pd.DataFrame(training_losses)
    if not loss_frame.empty:
        profile_frame = profile_frame.merge(loss_frame, on="step", how="left")
    return with_run_metadata(profile_frame, run_status, last_completed_step)


def write_partial_results(last_completed_step):
    if not checkpoint_frames and not profile_history:
        return
    RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    if checkpoint_frames:
        build_run_results(
            run_status="running_partial",
            last_completed_step=last_completed_step,
        ).to_csv(partial_results_path, index=False)
    if profile_history:
        build_profile_results(
            run_status="running_partial",
            last_completed_step=last_completed_step,
        ).to_csv(partial_profile_path, index=False)


def write_model_checkpoint(path, step, label):
    CHECKPOINT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "run_id": RUN_ID,
            "step": step,
            "label": label,
            "run_config": RUN_CONFIG,
            "model_state_dict": checkpoint_states[step],
        },
        path,
    )
    print(f"saved {label} model checkpoint: {path}")


def load_resume_checkpoint(model):
    if RESUME_CHECKPOINT_PATH is None:
        return 0
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    resume_step = int(checkpoint["step"])
    print(f"loaded resume checkpoint: {RESUME_CHECKPOINT_PATH} at step {resume_step}")
    return resume_step


model = make_model()
start_step = load_resume_checkpoint(model)
optimizer = schedulefree.AdamWScheduleFree(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
prior = SyntheticWorldDataLoader(
    num_steps=NUM_STEPS,
    batch_size=BATCH_SIZE,
    seed=RANDOM_SEED,
    generator=TRAIN_GENERATOR,
    allow_ollama=WAG_ALLOW_OLLAMA,
    ollama_model=WAG_OLLAMA_MODEL,
    hard_negative_rate=HARD_NEGATIVE_RATE,
    hard_negative_contract_weights=HARD_NEGATIVE_CONTRACT_WEIGHTS,
)

checkpoint_states = {start_step: clone_state_dict(model)}
checkpoint_frames = []
training_losses = []
profile_history = []
best_mean_pr_auc = -float("inf")
last_completed_step = start_step
interrupted = False

partial_results_path = RESULTS_DIRECTORY / f"{RUN_ID}.partial.csv"
partial_profile_path = RESULTS_DIRECTORY / f"{RUN_ID}.profile.partial.csv"
run_results_path = RESULTS_DIRECTORY / f"{RUN_ID}.csv"
profile_results_path = RESULTS_DIRECTORY / f"{RUN_ID}.profile.csv"
latest_checkpoint_path = CHECKPOINT_DIRECTORY / f"{RUN_ID}_latest.pt"
best_checkpoint_path = CHECKPOINT_DIRECTORY / f"{RUN_ID}_best.pt"

optimizer.eval()
checkpoint_frames.append(checkpoint_frame(model, start_step))
optimizer.train()
write_partial_results(last_completed_step)

prior_iter = iter(prior)
for _ in range(start_step):
    next(prior_iter)
if start_step:
    print(f"advanced synthetic generator by {start_step} batches")
try:
    for step in range(start_step + 1, NUM_STEPS + 1):
        step_start = time.perf_counter()

        generate_start = time.perf_counter()
        full_data = next(prior_iter)
        generate_seconds = time.perf_counter() - generate_start

        model.train()
        optimizer.train()
        optimizer.zero_grad()

        tokenize_start = time.perf_counter()
        tokenized_cells = move_to_device(
            tokenizer(full_data["records"], full_data["field_types"])
        )
        targets = full_data["adjacency"].to(device).float()
        tokenize_seconds = time.perf_counter() - tokenize_start

        forward_start = time.perf_counter()
        logits = model(tokenized_cells)
        mask = torch.triu(torch.ones_like(targets, dtype=torch.bool), diagonal=1)
        pair_logits = logits[mask]
        pair_labels = targets[mask]
        num_positive = pair_labels.sum()
        num_negative = pair_labels.numel() - num_positive
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            pair_logits,
            pair_labels,
            pos_weight=num_negative / num_positive
            if POS_WEIGHT is None
            else torch.tensor(POS_WEIGHT, device=pair_logits.device),
        )
        forward_loss_seconds = time.perf_counter() - forward_start

        backward_start = time.perf_counter()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        backward_step_seconds = time.perf_counter() - backward_start

        training_losses.append(
            {"step": step, "weighted_training_loss": float(loss.detach().cpu())}
        )

        checkpoint_seconds = 0.0
        eval_seconds = 0.0
        did_eval = step % EVAL_EVERY == 0 or step == NUM_STEPS
        if did_eval:
            eval_start = time.perf_counter()
            model.eval()
            optimizer.eval()
            checkpoint_start = time.perf_counter()
            checkpoint_states[step] = clone_state_dict(model)
            write_model_checkpoint(latest_checkpoint_path, step, "latest")
            checkpoint_seconds = time.perf_counter() - checkpoint_start
            eval_checkpoint_frame = checkpoint_frame(model, step)
            checkpoint_frames.append(eval_checkpoint_frame)
            current_mean_pr_auc = float(eval_checkpoint_frame["pr_auc"].mean())
            checkpoint_start = time.perf_counter()
            if current_mean_pr_auc > best_mean_pr_auc:
                best_mean_pr_auc = current_mean_pr_auc
                write_model_checkpoint(best_checkpoint_path, step, "best")
            checkpoint_seconds += time.perf_counter() - checkpoint_start
            eval_seconds = time.perf_counter() - eval_start
            optimizer.train()

        step_seconds = time.perf_counter() - step_start
        profile_row = {
            "step": step,
            "generate_seconds": generate_seconds,
            "tokenize_seconds": tokenize_seconds,
            "forward_loss_seconds": forward_loss_seconds,
            "backward_step_seconds": backward_step_seconds,
            "checkpoint_seconds": checkpoint_seconds,
            "eval_seconds": eval_seconds,
            "step_seconds": step_seconds,
            "rss_mb": current_rss_mb(),
            "device_memory_mb": current_device_memory_mb(),
            "did_eval": did_eval,
            **batch_shape_summary(full_data),
        }
        profile_history.append(profile_row)
        print(
            f"step {step:04d} | generate {generate_seconds:7.2f}s | "
            f"tokenize {tokenize_seconds:6.2f}s | train "
            f"{forward_loss_seconds + backward_step_seconds:6.2f}s | "
            f"eval {eval_seconds:6.2f}s | rss {profile_row['rss_mb']:7.1f} MB"
        )
        last_completed_step = step
        if did_eval:
            write_partial_results(last_completed_step)
except KeyboardInterrupt:
    interrupted = True
    print(f"interrupted after step {last_completed_step}")

run_status = "interrupted" if interrupted else "completed"
if interrupted:
    if checkpoint_frames:
        run_results = build_run_results(run_status, last_completed_step)
        run_results.to_csv(partial_results_path, index=False)
    if profile_history:
        profile_results = build_profile_results(run_status, last_completed_step)
        profile_results.to_csv(partial_profile_path, index=False)
    print(f"saved partial run results: {partial_results_path}")
    print(f"saved partial profile results: {partial_profile_path}")
else:
    run_results = build_run_results(run_status, last_completed_step)
    profile_results = build_profile_results(run_status, last_completed_step)
    RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
    if run_results_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing run: {run_results_path}")
    if profile_results_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing profile run: {profile_results_path}"
        )
    run_results.to_csv(run_results_path, index=False)
    profile_results.to_csv(profile_results_path, index=False)
    partial_results_path.unlink(missing_ok=True)
    partial_profile_path.unlink(missing_ok=True)
    print(f"saved run results: {run_results_path}")
    print(f"saved profile results: {profile_results_path}")

history = pd.concat(checkpoint_frames, ignore_index=True)
display(history.tail(8))


## Per-Benchmark Learning Curves

Compare ranking, fixed-threshold classification, oracle threshold headroom, and clustering across training.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for benchmark_name, frame in history.groupby("name"):
    axes[0, 0].plot(frame["step"], frame["pr_auc"], marker="o", label=benchmark_name)
    axes[0, 1].plot(frame["step"], frame["pair_f1"], marker="o", label=benchmark_name)
    axes[1, 0].plot(
        frame["step"], frame["best_pair_f1"], marker="o", label=benchmark_name
    )
    axes[1, 1].plot(
        frame["step"], frame["adjusted_rand"], marker="o", label=benchmark_name
    )

axes[0, 0].set_title("PR-AUC")
axes[0, 1].set_title("Pair F1 at threshold 0.5")
axes[1, 0].set_title("Oracle best pair F1")
axes[1, 1].set_title("Adjusted Rand")
for ax in axes.flat:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    ax.set_xlabel("training step")
plt.tight_layout()
plt.show()

thresholds = history.pivot(index="step", columns="name", values="best_pair_threshold")
thresholds.plot(figsize=(11, 4), marker="o", title="Oracle best threshold by benchmark")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
best_steps = (
    history.loc[
        history.groupby("name")["pr_auc"].idxmax(),
        ["name", "step", "pr_auc", "pair_f1", "best_pair_f1", "adjusted_rand"],
    ]
    .set_index("name")
    .sort_index()
)
display(best_steps)

final_step = int(history["step"].max())
best_rows = history.loc[history.groupby("name")["pr_auc"].idxmax()].set_index("name")
final_rows = history[history["step"] == final_step].set_index("name")
collaborator_summary = pd.DataFrame(
    {
        "best_step": best_rows["step"],
        "prevalence": best_rows["prevalence"],
        "best_pr_auc": best_rows["pr_auc"],
        "final_pr_auc": final_rows["pr_auc"],
        "best_pair_f1": best_rows["best_pair_f1"],
        "final_pair_f1": final_rows["pair_f1"],
        "best_adjusted_rand": best_rows["adjusted_rand"],
        "final_adjusted_rand": final_rows["adjusted_rand"],
    }
).sort_index()
display(collaborator_summary)

mean_history = history.groupby("step").mean(numeric_only=True)
best_mean_step = int(mean_history["pr_auc"].idxmax())
print(f"best mean PR-AUC checkpoint: {best_mean_step}")
print(f"final checkpoint: {final_step}")
display(mean_history.loc[[best_mean_step, final_step]])


def save_model_checkpoint(step, label):
    CHECKPOINT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    checkpoint_path = CHECKPOINT_DIRECTORY / f"{RUN_ID}_{label}_step{step:04d}.pt"
    torch.save(
        {
            "run_id": RUN_ID,
            "step": step,
            "label": label,
            "run_config": RUN_CONFIG,
            "model_state_dict": checkpoint_states[step],
        },
        checkpoint_path,
    )
    print(f"saved {label} model checkpoint: {checkpoint_path}")
    return checkpoint_path


if SAVE_MODELS:
    if SAVE_BEST_MODEL:
        save_model_checkpoint(best_mean_step, "best")
    if SAVE_LAST_MODEL and final_step != best_mean_step:
        save_model_checkpoint(final_step, "last")
    elif SAVE_LAST_MODEL:
        print("best and last checkpoint are the same step; saved one checkpoint")


## Pair-Level Diagnostics

Build detailed pair rows for one compact benchmark and one selected checkpoint. Change `ANALYSIS_STEP` to compare the best aggregate checkpoint with the final checkpoint.

In [ ]:
ANALYSIS_STEP = best_mean_step

analysis_model = make_model()
if PAIR_ANALYSIS_CHECKPOINT_PATH is None:
    analysis_model.load_state_dict(checkpoint_states[ANALYSIS_STEP])
else:
    checkpoint = torch.load(PAIR_ANALYSIS_CHECKPOINT_PATH, map_location=device)
    analysis_model.load_state_dict(checkpoint["model_state_dict"])
    ANALYSIS_STEP = checkpoint["step"]
    print(f"loaded pair-analysis checkpoint: {PAIR_ANALYSIS_CHECKPOINT_PATH}")
analysis_linker = NanoERPFNLinker(analysis_model, tokenizer, device=device)
analysis_benchmark = benchmark_by_name[PAIR_ANALYSIS_BENCHMARK]
analysis_linker.fit(analysis_benchmark["records"], analysis_benchmark["field_types"])


def build_pair_frame(linker, benchmark):
    records = benchmark["records"].reset_index(drop=True)
    entity_ids = benchmark["entity_ids"]
    left_indices, right_indices = np.triu_indices(len(records), k=1)
    left = records.iloc[left_indices].reset_index(drop=True)
    right = records.iloc[right_indices].reset_index(drop=True)
    equal_cells = left.eq(right) | (left.isna() & right.isna())
    probabilities = linker.adjacency_proba_[left_indices, right_indices]

    pairs = pd.DataFrame(
        {
            "left_index": left_indices,
            "right_index": right_indices,
            "left_entity_id": entity_ids[left_indices],
            "right_entity_id": entity_ids[right_indices],
            "target": entity_ids[left_indices] == entity_ids[right_indices],
            "probability": probabilities,
            "prediction": probabilities >= linker.threshold,
            "matching_fields": equal_cells.sum(axis=1),
            "differing_fields": equal_cells.apply(
                lambda row: ", ".join(row.index[~row]), axis=1
            ),
        }
    )
    for column in records:
        pairs[f"left_{column}"] = left[column]
        pairs[f"right_{column}"] = right[column]
    return pairs


pairs = build_pair_frame(analysis_linker, analysis_benchmark)
print(
    f"benchmark: {PAIR_ANALYSIS_BENCHMARK}; checkpoint: {ANALYSIS_STEP}; pairs: {len(pairs):,}"
)
display(pairs.head())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
bins = np.linspace(0, 1, 31)
axes[0].hist(
    pairs.loc[~pairs["target"], "probability"],
    bins=bins,
    alpha=0.65,
    density=True,
    label="negative",
)
axes[0].hist(
    pairs.loc[pairs["target"], "probability"],
    bins=bins,
    alpha=0.65,
    density=True,
    label="positive",
)
axes[0].set(
    title="Pair-probability distributions", xlabel="probability", ylabel="density"
)
axes[0].legend()

precision, recall, thresholds = precision_recall_curve(
    pairs["target"], pairs["probability"]
)
threshold_f1 = (
    2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
)
best_threshold_index = np.argmax(threshold_f1)
best_threshold = thresholds[best_threshold_index]
axes[1].plot(thresholds, precision[:-1], label="precision")
axes[1].plot(thresholds, recall[:-1], label="recall")
axes[1].plot(thresholds, threshold_f1, label="F1")
axes[1].axvline(0.5, linestyle="--", color="black", label="fixed 0.5")
axes[1].axvline(
    best_threshold,
    linestyle=":",
    color="black",
    label=f"best F1 {best_threshold:.3f}",
)
axes[1].set(title="Threshold behavior", xlabel="threshold", ylabel="metric")
axes[1].legend(fontsize=8)

plot_rng = np.random.default_rng(0)
max_negative_points_per_group = 500
plot_groups = []
group_counts = pairs.groupby(["matching_fields", "target"]).size()
for (matching_fields, target), group in pairs.groupby(["matching_fields", "target"]):
    if not target and len(group) > max_negative_points_per_group:
        group = group.sample(max_negative_points_per_group, random_state=0)
    group = group[["probability"]].copy()
    group["target"] = target
    group["x"] = matching_fields + (-0.12 if not target else 0.12)
    group["x"] += plot_rng.uniform(-0.08, 0.08, len(group))
    plot_groups.append(group)
plot_pairs = pd.concat(plot_groups, ignore_index=True)
for target, group in plot_pairs.groupby("target"):
    axes[2].scatter(
        group["x"],
        group["probability"],
        s=10,
        alpha=0.25,
        label="positive" if target else "negative",
    )
for (matching_fields, target), count in group_counts.items():
    axes[2].annotate(
        f"n={count}",
        (matching_fields + (-0.12 if not target else 0.12), 1.01),
        ha="center",
        va="bottom",
        fontsize=7,
        rotation=90,
    )
axes[2].set(
    title="Probability by target and matching fields",
    xlabel="number of exactly matching fields",
    ylabel="probability",
    ylim=(-0.02, 1.15),
)
axes[2].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

display(
    pairs.groupby(["matching_fields", "target"])["probability"].agg(
        ["count", "mean", "median", "min", "max"]
    )
)

## Confidence Review

Inspect confidently wrong pairs alongside confidently correct pairs. The displayed rows include both records and their differing-field summary.

In [ ]:
DISPLAY_COLUMNS = [
    "probability",
    "target",
    "matching_fields",
    "differing_fields",
    *[
        column
        for column in pairs
        if column.startswith("left_") or column.startswith("right_")
    ],
]


def show_pairs(title, frame, ascending, n=10):
    print(title)
    display(
        frame.sort_values("probability", ascending=ascending)[DISPLAY_COLUMNS].head(n)
    )


show_pairs(
    "Highest-confidence false positives", pairs[~pairs["target"]], ascending=False
)
show_pairs("Lowest-confidence true positives", pairs[pairs["target"]], ascending=True)
show_pairs("Highest-confidence true positives", pairs[pairs["target"]], ascending=False)
show_pairs("Lowest-confidence true negatives", pairs[~pairs["target"]], ascending=True)

In [ ]:
field_pattern_summary = (
    pairs.groupby(["target", "differing_fields"])["probability"]
    .agg(["count", "mean", "median", "max"])
    .sort_values(["target", "count"], ascending=[False, False])
)
display(field_pattern_summary.head(30))

## Clustering Failure Review

Connected components can merge many true entities through a small number of false-positive edges. Compare true and predicted component sizes, then inspect the strongest cross-entity predicted edges.

In [ ]:
cluster_frame = pd.DataFrame(
    {
        "true_entity": analysis_benchmark["entity_ids"],
        "predicted_component": analysis_linker.labels_,
    }
)
true_sizes = cluster_frame["true_entity"].value_counts().rename("true_cluster_size")
predicted_sizes = (
    cluster_frame["predicted_component"]
    .value_counts()
    .rename("predicted_component_size")
)
display(pd.concat([true_sizes.describe(), predicted_sizes.describe()], axis=1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
true_sizes.value_counts().sort_index().plot.bar(
    ax=axes[0], title="True cluster-size counts"
)
predicted_sizes.value_counts().sort_index().plot.bar(
    ax=axes[1], title="Predicted component-size counts"
)
plt.tight_layout()
plt.show()

predicted_false_edges = pairs[(~pairs["target"]) & pairs["prediction"]]
show_pairs(
    "Strongest false-positive component-merging edges",
    predicted_false_edges,
    ascending=False,
    n=20,
)

## Questions for the Next Pass

1. Which checkpoint gives the best ranking on each benchmark, and do those checkpoints disagree?
2. Which differing-field patterns produce low-confidence true matches?
3. Which shared values produce high-confidence false positives?
4. Does the model mostly fail from ranking errors, threshold choice, or connected-component chaining?
5. Which observed failures should drive DGP changes, threshold-transfer work, or model changes?